# arc3-cwm-backtest-coder - does a CODER model write better world models?

Free run, **no submission quota**.

Swaps only the model. Same 12 segments, same prompt, same calibration as the
`Qwen3.8-Flash-Next-NVFP4` arms (null floor **0/61**, proven ceiling
**58/61** via a generated lookup-table oracle).

| arm | passed |
|---|---|
| Flash-Next, thinking, 16k | 1/12 (but 27/27 replies truncated) |
| Flash-Next, no thinking, 8k | 0/12 (4 loaded and ran) |
| **Qwen3-Coder-30B-A3B** | this run |

No games are played, so this kernel needs no ARC runtime, no solver bundle
and no vLLM -- only the segments, the package and the model.


In [ ]:

# ============================================================
# Environment + input validation. Cheap, and FIRST.
# A previous run in this arm spent a 7.5h queue wait plus 14 minutes of
# GPU to discover a missing filename; everything checkable without the
# model is checked here.
# ============================================================
import glob, json, os, subprocess, sys, time
from pathlib import Path

T0 = time.time()
print(subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader"],
    capture_output=True, text=True).stdout.strip() or "nvidia-smi unavailable", flush=True)

import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), flush=True)
assert torch.cuda.is_available(), "no GPU"
_p = torch.cuda.get_device_properties(0)
_vram = _p.total_memory / 1e9
print(f"gpu {_p.name} sm_{_p.major}{_p.minor} {_vram:.1f} GB "
      f"bf16={torch.cuda.is_bf16_supported()}", flush=True)

# Qwen3-Coder-30B-A3B is ~60 GB in bf16. v1 of this kernel was handed a
# Tesla T4 (15.6 GB) despite requesting NvidiaRtxPro6000, so device_map=
# "auto" silently offloaded to disk and died ~10 minutes later inside
# from_pretrained with a confusing missing-offload_folder ValueError.
#
# The cause was almost certainly this kernel being the only one of 17 in
# the repo WITHOUT `competition_sources` -- that attachment appears to gate
# the competition's premium hardware pool. Re-added; this assert is the
# backstop so a wrong card costs seconds, not a queue slot.
MIN_VRAM_GB = float(os.environ.get("CWM_MIN_VRAM_GB", "40"))
assert _vram >= MIN_VRAM_GB, (
    f"got {_p.name} with {_vram:.1f} GB, need >= {MIN_VRAM_GB} GB for a 30B model. "
    "Check that kernel-metadata.json still has competition_sources AND "
    "machine_shape=NvidiaRtxPro6000 -- dropping the former silently downgrades "
    "the card even when the latter is set."
)


def find_input_dir(name):
    base = Path("/kaggle/input")
    for path in base.rglob("*"):
        if path.is_dir() and path.name == name:
            return path
    raise RuntimeError(f"{name!r} not found under /kaggle/input")


DATA_DIR = find_input_dir("cwm-backtest")
sys.path.insert(0, str(DATA_DIR))
os.environ["ARC3_CWM_ENGINE_DIR"] = str(DATA_DIR)
print("backtest data:", DATA_DIR, flush=True)

# Kaggle decompresses an uploaded .gz, so accept either name.
_cands = ["cwm_segments.json", "cwm_segments.json.gz"]
SEGMENTS_PATH = next((DATA_DIR / c for c in _cands if (DATA_DIR / c).is_file()), None)
if SEGMENTS_PATH is None:
    raise FileNotFoundError(
        f"no segments file in {DATA_DIR}; tried {_cands}; dir holds "
        f"{sorted(p.name for p in DATA_DIR.iterdir())}"
    )
print("segments file:", SEGMENTS_PATH.name, flush=True)

# Locate the coder model by its config.json, not a guessed path.
def find_model_dir(keyword):
    hits = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/config.json", recursive=True)
            if keyword in p.lower()]
    return sorted(hits, key=len)[0] if hits else None


CODER_MODEL_DIR = find_model_dir("qwen3-coder") or find_model_dir("coder") or find_model_dir("qwen")
print("CODER_MODEL_DIR:", CODER_MODEL_DIR, flush=True)
assert CODER_MODEL_DIR, "no coder model mounted -- check kernel-metadata model_sources"

os.environ["LLM_BACKEND"] = "transformers"
os.environ["CODER_MODEL_DIR"] = CODER_MODEL_DIR

from arc3_cwm.determinism import census
from arc3_cwm.oracle import verify_oracle
from arc3_cwm.serialize import load_segments

_segs = load_segments(SEGMENTS_PATH)
print(f"loaded {len(_segs)} segments from {len({s.game_id for s in _segs})} games", flush=True)
_det = census(_segs)
print(_det.summary(), flush=True)
_oracle_ok = sum(1 for s in _segs if verify_oracle(s)[0])
print(f"positive control: oracle replays {_oracle_ok}/{len(_segs)} segments", flush=True)
assert _oracle_ok > 0, (
    "the oracle cannot pass a single segment -- the harness could not report a "
    "pass even if the model produced one, so any result would be meaningless"
)
print(f"INPUT VALIDATION PASSED in {time.time()-T0:.1f}s", flush=True)


In [ ]:

# ============================================================
# The measurement. Only the MODEL differs from the Flash-Next arms.
# ============================================================
from arc3_cwm.harness import BacktestConfig, run_segment
from arc3_cwm.report import build_report, per_game_table

MAX_SEGMENTS = int(os.environ.get("CWM_MAX_SEGMENTS", "12"))
MAX_ATTEMPTS = int(os.environ.get("CWM_MAX_ATTEMPTS", "3"))
MAX_TOKENS = int(os.environ.get("CWM_MAX_TOKENS", "8192"))
SOFT_DEADLINE_S = float(os.environ.get("CWM_SOFT_DEADLINE_S", str(6.0 * 3600)))

RESULTS_PATH = Path("/kaggle/working/cwm_coder_results.json")
SOURCES_DIR = Path("/kaggle/working/passing_models")

segments = load_segments(SEGMENTS_PATH)

# Round-robin across games, matching the Flash-Next arms exactly so the
# comparison is on identical segments -- not the alphabetical file order.
by_game = {}
for seg in segments:
    by_game.setdefault(seg.game_id, []).append(seg)
for group in by_game.values():
    group.sort(key=lambda s: s.level)
ordered, depth = [], 0
while len(ordered) < len(segments):
    added = False
    for game in sorted(by_game):
        if depth < len(by_game[game]):
            ordered.append(by_game[game][depth]); added = True
    if not added:
        break
    depth += 1
segments = ordered[:MAX_SEGMENTS]
print(f"pilot: {len(segments)} segments across "
      f"{len({s.game_id for s in segments})} games "
      f"(levels {sorted({s.level for s in segments})})", flush=True)

# If anything still spills, give it somewhere to go rather than raising.
os.environ.setdefault("CWM_OFFLOAD_DIR", "/kaggle/working/offload")
Path(os.environ["CWM_OFFLOAD_DIR"]).mkdir(parents=True, exist_ok=True)

print("loading the coder model (this is the slow part)...", flush=True)
_t = time.time()
from llm_engine.llm_client import make_client

client = make_client("coder")
print(f"model loaded in {time.time()-_t:.1f}s", flush=True)

# Preflight that exercises the REAL shape of the task. The Flash-Next run's
# preflight used a short prompt, passed, and told us nothing -- every real
# prompt then behaved differently.
_probe = client.complete(
    "You write Python. Reply with only a code fence.",
    "Write a class named WorldModel with a predict(self, state, action_name, "
    "x=None, y=None) method returning (state, 0, False) and a goal_hint(self, "
    "state) method returning 0.0.",
    max_tokens=512,
)
print(f"preflight reply ({len(_probe)} chars):\n{_probe[:600]}", flush=True)
assert _probe.strip(), "model returned nothing -- do NOT trust any result from this run"
print("preflight contains a WorldModel class:", "class WorldModel" in _probe, flush=True)

config = BacktestConfig(max_attempts=MAX_ATTEMPTS, max_tokens=MAX_TOKENS)
results = []
started = time.time()


def _persist(partial):
    payload = build_report(results).as_dict()
    payload["determinism"] = _det.as_dict()
    payload["oracle_passing_segments"] = _oracle_ok
    payload["model_dir"] = CODER_MODEL_DIR
    payload["backend"] = "transformers"
    payload["max_tokens"] = MAX_TOKENS
    payload["partial"] = partial
    payload["segments_attempted"] = len(results)
    payload["segments_total"] = len(segments)
    RESULTS_PATH.write_text(json.dumps(payload, indent=2), encoding="utf-8")


for index, segment in enumerate(segments, start=1):
    if time.time() - started > SOFT_DEADLINE_S:
        print(f"soft deadline -- stopping with {len(results)}/{len(segments)}", flush=True)
        break
    result = run_segment(client, segment, config)
    results.append(result)
    print(f"[{index}/{len(segments)}] {segment.key:22s} n={len(segment):3d} "
          f"{result.outcome:14s} prefix={result.best_prefix:3d}/{len(segment):<3d} "
          f"att={result.attempts} {result.elapsed_s:7.1f}s", flush=True)
    _persist(partial=True)
    if result.source:
        SOURCES_DIR.mkdir(exist_ok=True)
        (SOURCES_DIR / f"coder_{result.segment_key.replace('/', '_')}.py").write_text(
            result.source, encoding="utf-8")

report = build_report(results)
print()
print(report.summary(), flush=True)
print()
print(per_game_table(results), flush=True)
_persist(partial=len(results) < len(segments))
print(f"\nwrote {RESULTS_PATH}", flush=True)
print(f"total wall clock: {time.time()-started:.1f}s", flush=True)
